# Chapter 2 minutes primary-results reconciliation — 2026-09-04

## tl;dr

The selected source is the **2026-09-01 all391 / chk2 checkpoint-50 release reconstruction diagnostic**.
Its complete panel contains **391 samples, 124 meetings (2009-01-28–2025-01-29), three models, ten replicates and 11,730 score rows**.
The six independently recomputed differences round to the current main-table values:
MPNet **0.005201 / 0.002365 / 0.007567** and BERTScore-F1 **0.003430 / 0.000833 / 0.004264**
for chk1−chk0 / chk2−chk1 / chk2−chk0.

All six paired t-tests and frozen meeting sign-flip tests reject at 1%, including the Holm-adjusted sign-flip tests.
**The two chk2−chk1 hierarchical bootstrap intervals include zero.** Their uncertainty must be reported separately from t-test or sign-flip significance.
The panel includes the training release and teacher-synthetic references; these results cannot support an external held-out generalization claim.

## Context & Methods

This notebook is a reproducible companion for the thesis correction. It reads local frozen outputs, checks file and row integrity,
recomputes the published means and paired t-tests, and preserves the frozen bootstrap and randomization results without generating or scoring new text.
Run top-to-bottom from the repository root or a descendant directory with Python, NumPy, pandas, SciPy, nbformat, nbclient and ipykernel.
The final cell rewrites only the paired-results CSV and provenance JSON in `docs/Chapter2/Chapter2Results/`.

### Key Assumptions

- `raw_best_effort` scores use recovered text, retaining invalid delivery rows; no outcome-dependent exclusions are added.
- Average ten replicates within each sample, then sample rows within each meeting, then weight all 124 meetings equally.
  Balanced replicate coverage makes this equivalent to the frozen implementation's within-meeting rows → replicates → meetings order.
- The t-test uses the 124 paired meeting differences, two-sided Student t inference, df = 123, and an unadjusted 95% interval.
  It conditions on the observed replicate averages and assumes independent meeting-level differences; it does not establish causality or remove temporal dependence.
- The frozen primary interval uses 1,000 split-stratified paired meeting and replicate bootstrap draws.
  The meeting sign-flip test conditions on the observed replicate averages, uses 100,000 seeded assignments and a plus-one correction;
  Holm adjusts the six primary sign-flip tests only. These procedures assess uncertainty differently and need not agree.
- Every relative percentage uses the corresponding **chk0 mean** as denominator, including chk2−chk1.
- Checkpoint lineage is recorded from the verified manifests. Model-weight directories are not rehashed and semantic scores are not regenerated here.

In [1]:
from pathlib import Path
import hashlib
import itertools
import json
import platform
from collections import Counter

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from IPython.display import display

RUN_REL = Path("output/evaluation/retrain_v2/paper_chk2_text_similarity_all391_chk0_chk1_chk2cp50_vllm_t06_p09_k10_b1000_v1_20260901")
NOTEBOOK_REL = Path("docs/Chapter2/review/minutes_primary_results_reconciliation_20260904.ipynb")
RESULT_REL = Path("docs/Chapter2/Chapter2Results/minutes_primary_meeting_paired_tests_20260904.csv")
PROVENANCE_REL = Path("docs/Chapter2/Chapter2Results/minutes_primary_results_provenance_20260904.json")
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / RUN_REL).is_dir()), None)
assert ROOT is not None, "Run from the repository root or a descendant directory containing the frozen source."
RUN = ROOT / RUN_REL
MODELS = ["chk0", "chk1", "chk2"]
METRICS = ["mpnet_cosine", "bertscore_f1"]
REPLICATES = list(range(10))
POLICY = "raw_best_effort"
SCHEME = "meeting_cluster_primary"
CONTRASTS = [("chk1", "chk0"), ("chk2", "chk1"), ("chk2", "chk0")]
PINNED_SCORE_MANIFEST_SHA256 = "f76666c99d5fb8c50fab637dff92d6cdb6c672d4db23fd9a238e298c6d949ede"
print("Source:", RUN_REL.as_posix())
print("Environment:", {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__})

Source: output/evaluation/retrain_v2/paper_chk2_text_similarity_all391_chk0_chk1_chk2cp50_vllm_t06_p09_k10_b1000_v1_20260901
Environment: {'python': '3.11.5', 'numpy': '1.26.4', 'pandas': '2.1.4', 'scipy': '1.11.1'}


## Data

### 1. Verify source bindings

The pinned root `score_manifest.json` anchors this audit. Check its payload seal, all artifact/input hashes,
the original evaluation manifest, scoring implementation, and release/selection bindings.
Historical absolute paths are mapped to this repository; no source artifact is modified.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, allow_nan=False,
                                    sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def load_jsonl(path):
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

def resolve_source(source):
    path = Path(source)
    if not path.is_absolute():
        return RUN / path if (RUN / path).exists() else ROOT / path
    # Rebase historical absolute paths while retaining their full repository suffix.
    marker = "/fomc_trainer/"
    assert marker in source, f"Unrecognized source path: {source}"
    return ROOT / source.split(marker, 1)[1]

hash_checks = []
def verify_binding(label, binding):
    path = resolve_source(binding["path"])
    digest = sha256_file(path)
    assert digest == binding["sha256"], f"SHA256 mismatch: {label}"
    if "bytes" in binding:
        assert path.stat().st_size == binding["bytes"], f"Byte-count mismatch: {label}"
    if "payload_sha256" in binding:
        payload = load_json(path)
        observed = canonical_sha({k: v for k, v in payload.items() if k != "integrity"})
        assert observed == binding["payload_sha256"], f"Payload mismatch: {label}"
    hash_checks.append({"binding": label, "path": path.relative_to(ROOT).as_posix(),
                        "sha256": digest, "bytes": path.stat().st_size, "verified": True})

verify_binding("pinned_score_manifest", {"path": "score_manifest.json", "sha256": PINNED_SCORE_MANIFEST_SHA256})
score_manifest = load_json(RUN / "score_manifest.json")
complete_manifest = load_json(RUN / "complete_evaluation_manifest.json")
evaluation_manifest = load_json(RUN / "evaluation_manifest.json")
for name, manifest in [("score", score_manifest), ("complete", complete_manifest)]:
    assert canonical_sha({k: v for k, v in manifest.items() if k != "integrity"}) == manifest["integrity"]["payload_sha256"], name
for group in ["artifacts", "inputs"]:
    for name, binding in score_manifest[group].items():
        verify_binding(f"score_manifest.{group}.{name}", binding)
verify_binding("scoring_implementation", score_manifest["implementation"])
verify_binding("source_evaluation_manifest", complete_manifest["inputs"]["source_evaluation_manifest"])
for name in ["release_manifest", "prompt_contract"]:
    verify_binding(f"evaluation_release.{name}", evaluation_manifest["release"][name])
verify_binding("checkpoint_selection", evaluation_manifest["checkpoint_selection"])
contract = score_manifest["score_contract"]
assert canonical_sha({k: v for k, v in contract.items() if k != "sha256"}) == contract["sha256"]
assert score_manifest["status"] == complete_manifest["status"] == "complete"
assert score_manifest["evaluation_role"] == "in_sample_training_release_reconstruction_diagnostic"
assert evaluation_manifest["scope"]["held_out_generalization_claim_allowed"] is False
assert score_manifest["training_release_evaluation_eligible"] is False
print(f"PASS: {len(hash_checks)} file bindings, both manifest payload seals and the score-contract seal.")
print("Evaluation role:", score_manifest["evaluation_role"])

PASS: 17 file bindings, both manifest payload seals and the score-contract seal.
Evaluation role: in_sample_training_release_reconstruction_diagnostic


### 2. Verify panel completeness and row provenance

The expected key is `(sample_id, model_id, replicate_id)`. Validate the full Cartesian product,
meeting/split assignments, finite raw scores, paired seeds, and prompt/reference hashes.

In [3]:
sample_rows = load_jsonl(RUN / "samples.jsonl")
score_rows = load_jsonl(RUN / "row_scores.jsonl")
samples = pd.DataFrame(sample_rows)
assert len(samples) == samples["sample_id"].nunique() == 391
assert samples["meeting_id"].nunique() == 124
assert samples.groupby("meeting_id")["split"].nunique().eq(1).all()
sample_lookup = {r["sample_id"]: r for r in sample_rows}
expected_keys = set(itertools.product(sample_lookup, MODELS, REPLICATES))
observed_keys = [(r["sample_id"], r["model_id"], r["replicate_id"]) for r in score_rows]
assert len(score_rows) == len(set(observed_keys)) == 11730
assert set(observed_keys) == expected_keys
assert sorted(r["score_row_number"] for r in score_rows) == list(range(1, 11731))
for sample in sample_rows:
    for field in ["prompt", "reference"]:
        assert hashlib.sha256(sample[field].encode("utf-8")).hexdigest() == sample[f"{field}_sha256"]
    assert canonical_sha(sample["prompt_token_ids"]) == sample["prompt_token_ids_sha256"]
    assert len(sample["prompt_token_ids"]) == sample["prompt_token_count"]
for row in score_rows:
    sample = sample_lookup[row["sample_id"]]
    for field in ["meeting_id", "split", "prompt_sha256", "reference_sha256"]:
        assert row[field] == sample[field], (row["sample_id"], field)
    assert row["score_contract_sha256"] == contract["sha256"]
    assert row["replicate_seed"] == evaluation_manifest["replicate_seeds"][row["replicate_id"]]
    assert all(np.isfinite(row[POLICY][metric]) for metric in METRICS)
flat_scores = pd.DataFrame([{**{k: r[k] for k in ["sample_id", "meeting_id", "split", "model_id", "replicate_id", "row_seed"]},
                            **r[POLICY]} for r in score_rows])
assert flat_scores.groupby(["sample_id", "replicate_id"])["row_seed"].nunique().eq(1).all()
coverage = {"samples": len(samples), "meetings": samples["meeting_id"].nunique(),
            "models": len(MODELS), "replicates": len(REPLICATES), "row_scores": len(score_rows),
            "meeting_date_min": samples["meeting_id"].min(), "meeting_date_max": samples["meeting_id"].max(),
            "split_rows": samples["split"].value_counts().to_dict(),
            "split_meetings": samples.groupby("split")["meeting_id"].nunique().to_dict()}
for name in ["samples", "meetings", "replicates", "row_scores", "split_rows", "split_meetings"]:
    assert coverage[name] == score_manifest["coverage"][name], name
assert Counter(r["model_id"] for r in score_rows) == {m: 3910 for m in MODELS}
assert coverage["meeting_date_min"] == "2009-01-28" and coverage["meeting_date_max"] == "2025-01-29"
print("PASS:", json.dumps(coverage, indent=2))
display(samples.groupby("split").agg(samples=("sample_id", "size"), meetings=("meeting_id", "nunique"),
                                    first_meeting=("meeting_id", "min"), last_meeting=("meeting_id", "max")))

PASS: {
  "samples": 391,
  "meetings": 124,
  "models": 3,
  "replicates": 10,
  "row_scores": 11730,
  "meeting_date_min": "2009-01-28",
  "meeting_date_max": "2025-01-29",
  "split_rows": {
    "train": 305,
    "test": 44,
    "validation": 42
  },
  "split_meetings": {
    "test": 13,
    "train": 98,
    "validation": 13
  }
}


,samples,meetings,first_meeting,last_meeting
split,,,,
test,44,13,2023-07-26,2025-01-29
train,305,98,2009-01-28,2021-11-03
validation,42,13,2021-12-15,2023-06-14


## Results

### 3. Recompute equal-meeting model means

Average the ten replicates per sample, then samples per meeting. Validate all six model/metric means against
`model_summary.csv` under `meeting_cluster_primary` / `raw_best_effort`.

In [4]:
sample_means = flat_scores.groupby(["sample_id", "meeting_id", "model_id"], sort=True)[METRICS].mean().reset_index()
meeting_means = sample_means.groupby(["meeting_id", "model_id"], sort=True)[METRICS].mean().reset_index()
assert len(sample_means) == 391 * 3 and len(meeting_means) == 124 * 3
means = meeting_means.groupby("model_id")[METRICS].mean().loc[MODELS]
frozen_model_summary = pd.read_csv(RUN / "model_summary.csv", float_precision="round_trip")
frozen_means = frozen_model_summary.query("bootstrap_scheme == @SCHEME and scoring_policy == @POLICY")
assert len(frozen_means) == 6
for row in frozen_means.itertuples():
    assert abs(means.loc[row.model_id, row.metric] - row.estimate) < 1e-12
print("PASS: all six means match the frozen summary within 1e-12.")
display(means.style.format("{:.9f}"))

PASS: all six means match the frozen summary within 1e-12.


,mpnet_cosine,bertscore_f1
model_id,,
chk0,0.936793348,0.957022279
chk1,0.941994795,0.960452510
chk2,0.944359874,0.961285783


### 4. Recompute paired t-tests

For meeting differences $d_i$, compute $\bar d$, $SE=s_d/\sqrt{124}$, $t=\bar d/SE$,
and $\bar d\pm t_{0.975,123}SE$. Verify independently against SciPy's paired test.
These t-test intervals and unadjusted p-values are supplementary to the frozen primary bootstrap and sign-flip outputs.

In [5]:
expected_rounded = {
    "mpnet_cosine": ["0.005201", "0.002365", "0.007567"],
    "bertscore_f1": ["0.003430", "0.000833", "0.004264"],
}
paired_records = []
difference_vectors = {}
for metric in METRICS:
    wide = meeting_means.pivot(index="meeting_id", columns="model_id", values=metric).sort_index()
    assert wide.shape == (124, 3) and wide.notna().all().all()
    for contrast_index, (after, before) in enumerate(CONTRASTS):
        differences = (wide[after] - wide[before]).to_numpy()
        contrast_id = f"{after}_minus_{before}"
        difference_vectors[(contrast_id, metric)] = differences
        n = len(differences)
        delta = float(differences.mean())
        sd = float(differences.std(ddof=1))
        se = sd / np.sqrt(n)
        t_stat = delta / se
        p_value = float(2 * stats.t.sf(abs(t_stat), df=n - 1))
        half_width = float(stats.t.ppf(0.975, df=n - 1) * se)
        independent_test = stats.ttest_rel(wide[after], wide[before])
        assert np.isclose(t_stat, independent_test.statistic, atol=1e-12, rtol=1e-12)
        assert np.isclose(p_value, independent_test.pvalue, atol=1e-14, rtol=1e-12)
        assert f"{delta:.6f}" == expected_rounded[metric][contrast_index]
        paired_records.append({"contrast_id": contrast_id, "metric": metric, "before_model": before,
                               "after_model": after, "scoring_policy": POLICY, "meetings": n, "df": n - 1,
                               "chk0_mean": float(means.loc["chk0", metric]), "before_mean": float(wide[before].mean()),
                               "after_mean": float(wide[after].mean()), "estimate": delta,
                               "relative_percent_of_chk0": 100 * delta / means.loc["chk0", metric],
                               "paired_difference_sd": sd, "paired_t_se": float(se), "paired_t_statistic": float(t_stat),
                               "paired_t_p_two_sided_unadjusted": p_value,
                               "paired_t_ci_lower_95": delta - half_width, "paired_t_ci_upper_95": delta + half_width})
paired_results = pd.DataFrame(paired_records)
print("PASS: all six differences round to the main table; df = 123 for every test.")
display(paired_results[["contrast_id", "metric", "estimate", "relative_percent_of_chk0", "paired_t_statistic",
                        "paired_t_p_two_sided_unadjusted", "paired_t_ci_lower_95", "paired_t_ci_upper_95"]]
        .style.format(precision=9))

PASS: all six differences round to the main table; df = 123 for every test.


,contrast_id,metric,estimate,relative_percent_of_chk0,paired_t_statistic,paired_t_p_two_sided_unadjusted,paired_t_ci_lower_95,paired_t_ci_upper_95
0,chk1_minus_chk0,mpnet_cosine,0.005201446,0.555239464,5.886025160,0.000000035,0.003452228,0.006950665
1,chk2_minus_chk1,mpnet_cosine,0.002365079,0.252465419,2.845227480,0.005200321,0.000719682,0.004010476
2,chk2_minus_chk0,mpnet_cosine,0.007566526,0.807704884,8.057126962,0.000000000,0.005707616,0.009425436
3,chk1_minus_chk0,bertscore_f1,0.003430231,0.358427526,12.440414280,0.000000000,0.002884435,0.003976028
4,chk2_minus_chk1,bertscore_f1,0.000833273,0.087069319,2.935921701,0.003970158,0.000271469,0.001395077
5,chk2_minus_chk0,bertscore_f1,0.004263504,0.445496846,12.915767335,0.000000000,0.003610090,0.004916918


### 5. Preserve and verify frozen bootstrap intervals and sign-flip inference

Recalculate percentile endpoints from the 1,000 saved paired bootstrap draws, retaining their paired replicate resampling.
Independently rerun only the lightweight meeting sign-flip arithmetic with each frozen seed, then verify its six-test Holm correction.
No model generation, semantic scoring, or new bootstrap sampling is performed.

In [6]:
frozen_pairwise = pd.read_csv(RUN / "pairwise_contrasts.csv", float_precision="round_trip")
primary = frozen_pairwise.query("bootstrap_scheme == @SCHEME and scoring_policy == @POLICY and aggregate_scope == 'pooled_k10'")
assert len(primary) == 6 and not primary.duplicated(["contrast_id", "metric"]).any()
bootstrap_rows = load_jsonl(RUN / "bootstrap_draws.jsonl")
assert len(bootstrap_rows) == 1000
assert sorted(row["draw_id"] for row in bootstrap_rows) == list(range(1000))
for row in primary.itertuples():
    matching = (paired_results["contrast_id"] == row.contrast_id) & (paired_results["metric"] == row.metric)
    assert abs(paired_results.loc[matching, "estimate"].iloc[0] - row.estimate) < 1e-12
    draws = []
    for draw in bootstrap_rows:
        values = draw["views"][SCHEME][POLICY]
        replicate_indices = np.asarray(draw["replicate_resample_indices"], dtype=int)
        per_k_delta = np.asarray(values[row.after_model][row.metric]["per_k"]) - np.asarray(values[row.before_model][row.metric]["per_k"])
        pooled_delta = float(per_k_delta[replicate_indices].mean())
        recorded_delta = values[row.after_model][row.metric]["pooled_k10"] - values[row.before_model][row.metric]["pooled_k10"]
        assert abs(pooled_delta - recorded_delta) < 1e-12
        draws.append(pooled_delta)
    lower, upper = np.quantile(draws, [0.025, 0.975], method="linear")
    assert np.allclose([lower, upper], [row.ci_lower, row.ci_upper], atol=1e-12, rtol=0)
    differences = difference_vectors[(row.contrast_id, row.metric)]
    seed_payload = f"{score_manifest['execution']['bootstrap_seed'] + 3}:{row.contrast_id}:{row.metric}"
    seed = int.from_bytes(hashlib.sha256(seed_payload.encode()).digest()[:4], "big")
    assert seed == int(row.sign_flip_seed)
    rng = np.random.default_rng(seed)
    assignments = int(row.sign_flip_assignments)
    extreme = 0
    for offset in range(0, assignments, 10000):
        count = min(10000, assignments - offset)
        signs = rng.choice(np.asarray([-1.0, 1.0]), size=(count, len(differences)))
        simulated = np.abs((signs * differences).mean(axis=1))
        extreme += int(np.count_nonzero(simulated >= abs(differences.mean()) - 1e-15))
    sign_flip_p = (extreme + 1.0) / (assignments + 1.0)
    assert np.isclose(sign_flip_p, row.p_value, atol=1e-14, rtol=0)
    for name, value in {
        "frozen_bootstrap_ci_lower_95": row.ci_lower, "frozen_bootstrap_ci_upper_95": row.ci_upper,
        "frozen_bootstrap_draws": int(row.bootstrap_draws),
        "meeting_sign_flip_p_two_sided": row.p_value, "meeting_sign_flip_holm_adjusted_p": row.holm_adjusted_p,
        "meeting_sign_flip_holm_family_size": int(row.holm_family_size),
        "meeting_sign_flip_assignments": assignments, "meeting_sign_flip_seed": seed,
    }.items():
        paired_results.loc[matching, name] = value
p_values = paired_results["meeting_sign_flip_p_two_sided"].to_numpy()
order = np.argsort(p_values, kind="stable")
holm = np.empty(len(p_values))
holm[order] = np.minimum(1, np.maximum.accumulate((len(p_values) - np.arange(len(p_values))) * p_values[order]))
assert np.allclose(holm, paired_results["meeting_sign_flip_holm_adjusted_p"], atol=1e-14, rtol=0)
for name in ["frozen_bootstrap_draws", "meeting_sign_flip_holm_family_size", "meeting_sign_flip_assignments", "meeting_sign_flip_seed"]:
    paired_results[name] = paired_results[name].astype("int64")
paired_results["frozen_bootstrap_ci_includes_zero"] = (paired_results["frozen_bootstrap_ci_lower_95"] <= 0) & (paired_results["frozen_bootstrap_ci_upper_95"] >= 0)
assert paired_results["frozen_bootstrap_ci_includes_zero"].equals(paired_results["contrast_id"].eq("chk2_minus_chk1"))
assert paired_results["paired_t_p_two_sided_unadjusted"].lt(0.05).all()
assert paired_results["meeting_sign_flip_holm_adjusted_p"].lt(0.05).all()
print("PASS: six bootstrap intervals, seeded sign-flip p-values and Holm adjustments reproduce the frozen outputs.")
display(paired_results[["contrast_id", "metric", "frozen_bootstrap_ci_lower_95", "frozen_bootstrap_ci_upper_95",
                        "frozen_bootstrap_ci_includes_zero", "meeting_sign_flip_p_two_sided", "meeting_sign_flip_holm_adjusted_p"]]
        .style.format(precision=9))

PASS: six bootstrap intervals, seeded sign-flip p-values and Holm adjustments reproduce the frozen outputs.


,contrast_id,metric,frozen_bootstrap_ci_lower_95,frozen_bootstrap_ci_upper_95,frozen_bootstrap_ci_includes_zero,meeting_sign_flip_p_two_sided,meeting_sign_flip_holm_adjusted_p
0,chk1_minus_chk0,mpnet_cosine,0.002132312,0.008000313,False,0.000010000,0.000059999
1,chk2_minus_chk1,mpnet_cosine,-0.000555840,0.005238744,True,0.005239948,0.005239948
2,chk2_minus_chk0,mpnet_cosine,0.004405331,0.011333017,False,0.000010000,0.000059999
3,chk1_minus_chk0,bertscore_f1,0.002551583,0.004352870,False,0.000010000,0.000059999
4,chk2_minus_chk1,bertscore_f1,-0.000089703,0.001823158,True,0.002469975,0.004939951
5,chk2_minus_chk0,bertscore_f1,0.003268496,0.005300773,False,0.000010000,0.000059999


## Takeaways

The current main-table point estimates are traceable to the complete 124-meeting all391 panel. A 128-meeting external-evaluation population,
1993–2008 date range, or chk2−chk1 effects of 0.061446 / 0.057130 do not describe this selected output.
Chk1 improves both raw semantic metrics relative to chk0; chk2 adds a smaller positive increment.
For that increment, the meeting-level t and Holm-adjusted sign-flip tests reject at 1%, while **both primary hierarchical bootstrap intervals cross zero**.
Report that method-dependent uncertainty and the reconstruction-only scope wherever interpreting the improvement.

The following export records the source hashes, checkpoint identities, sample coverage, means and six tests in separately named fields.
The `training_release_evaluation_eligible` flag is retained exactly as recorded; it does not grant external-evaluation status.
All checks here validate saved score arithmetic and provenance, not the semantic model computation or model-weight bytes.

In [7]:
def repo_relative_text(value):
    if isinstance(value, dict):
        return {k: repo_relative_text(v) for k, v in value.items()}
    if isinstance(value, list):
        return [repo_relative_text(v) for v in value]
    if isinstance(value, str) and "/fomc_trainer/" in value:
        return value.split("/fomc_trainer/", 1)[1]
    return value

result_path = ROOT / RESULT_REL
result_path.parent.mkdir(parents=True, exist_ok=True)
paired_results.to_csv(result_path, index=False, float_format="%.17g", lineterminator="\n")
provenance = {
    "schema_version": "chapter2-minutes-primary-results-reconciliation-v1",
    "reconciliation_date": "2026-09-04", "source_run": RUN_REL.as_posix(),
    "notebook": NOTEBOOK_REL.as_posix(), "results_csv": RESULT_REL.as_posix(),
    "results_csv_sha256": sha256_file(result_path),
    "validation_status": "all_executed_assertions_passed",
    "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__},
    "source_bindings": hash_checks,
    "score_manifest_payload_sha256": score_manifest["integrity"]["payload_sha256"],
    "score_contract_sha256": contract["sha256"],
    "coverage": coverage, "evaluation_role": score_manifest["evaluation_role"],
    "scope": evaluation_manifest["scope"],
    "training_release_evaluation_eligible": score_manifest["training_release_evaluation_eligible"],
    "models": repo_relative_text(evaluation_manifest["models"]),
    "model_fingerprints_recorded_not_rehashed": repo_relative_text(evaluation_manifest["model_fingerprints"]),
    "checkpoint_selection": repo_relative_text(evaluation_manifest["checkpoint_selection"]),
    "release": repo_relative_text(evaluation_manifest["release"]),
    "scoring_policy": POLICY, "reference": evaluation_manifest["scope"]["reference"],
    "aggregation": "replicate mean within sample; sample mean within meeting; equal meeting mean",
    "relative_percent_denominator": "metric-specific chk0 equal-meeting mean for every contrast",
    "paired_t": {"method": "two-sided Student paired t on 124 observed replicate-averaged meeting differences", "df": 123,
                 "confidence": 0.95, "multiplicity_adjusted": False},
    "frozen_bootstrap": {"scheme": SCHEME, "draws": 1000, "confidence": 0.95,
                         "method": "split-stratified paired meeting and paired replicate resampling; linear percentile intervals",
                         "verification": "recomputed endpoints from hash-verified frozen bootstrap_draws.jsonl"},
    "meeting_sign_flip": {"method": "two-sided Monte Carlo plus-one", "assignments": 100000,
                          "multiplicity": "Holm across 3 contrasts x 2 raw metrics", "family_size": 6,
                          "verification": "independently rerun using frozen seeds and raw-score meeting differences"},
    "model_means": means.reset_index().to_dict(orient="records"),
    "results": paired_results.to_dict(orient="records"),
    "limitations": ["Training-release reconstruction diagnostic with teacher-synthetic references; external held-out claims prohibited.",
                    "Both chk2-minus-chk1 hierarchical bootstrap intervals include zero despite significant paired t and sign-flip tests.",
                    "Meeting-level t and sign-flip inference conditions on observed replicate averages; temporal dependence is not modeled.",
                    "Model-weight directories not rehashed; generation and semantic scoring not rerun."],
}
(ROOT / PROVENANCE_REL).write_text(json.dumps(provenance, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
assert sha256_file(result_path) == provenance["results_csv_sha256"]
assert len(pd.read_csv(result_path)) == 6
print("Wrote:", RESULT_REL.as_posix())
print("Wrote:", PROVENANCE_REL.as_posix())
print("All assertions passed; source files were read only.")

Wrote: docs/Chapter2/Chapter2Results/minutes_primary_meeting_paired_tests_20260904.csv
Wrote: docs/Chapter2/Chapter2Results/minutes_primary_results_provenance_20260904.json
All assertions passed; source files were read only.
